[Reference](https://medium.com/@CodeWithYog/5e149994555b)

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
# 1) Load a light model
model = SentenceTransformer("all-MiniLM-L6-v2")
# 2) Your corpus
docs = [
    {"id":"1", "text":"Azure Functions scale on demand for event driven apps", "tag":"cloud"},
    {"id":"2", "text":"xUnit tests help catch regressions in .NET APIs", "tag":"testing"},
    {"id":"3", "text":"Vector databases store embeddings for semantic search", "tag":"ai"},
]
# 3) Embed
emb = model.encode([d["text"] for d in docs], normalize_embeddings=True).astype("float32")
# 4) Build HNSW index
d = emb.shape[1]
index = faiss.IndexHNSWFlat(d, 32)
index.hnsw.efConstruction = 200
index.add(emb)
# 5) Query
q = "How do I search by meaning not keywords?"
qv = model.encode([q], normalize_embeddings=True).astype("float32")
D, I = index.search(qv, k=2)
hits = [docs[i] for i in I[0]]
for r, score in zip(hits, D[0]):
    print(r["id"], r["text"], float(score))

# Index type
- HNSW. Great recall and fast search. Memory heavy. Strong general pick.
- IVF+PQ. Good for huge sets. Trades a bit of recall for low memory.
- Flat. Exact results. Fine for small sets or re-ranking top hits.

In [3]:
from typing import List, Dict
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import faiss
class Retriever:
    def __init__(self, docs: List[Dict]):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.re_ranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        self.docs = docs
        self.emb = self.model.encode([d["text"] for d in docs], normalize_embeddings=True).astype("float32")
        d = self.emb.shape[1]
        self.index = faiss.IndexHNSWFlat(d, 32)
        self.index.add(self.emb)
    def search(self, query: str, k: int = 10, top: int = 3):
        qv = self.model.encode([query], normalize_embeddings=True).astype("float32")
        _, I = self.index.search(qv, k)
        cands = [(i, self.docs[i]["text"]) for i in I[0]]
        pairs = [[query, t] for _, t in cands]
        scores = self.re_ranker.predict(pairs)
        order = np.argsort(-scores)[:top]
        return [cands[i][0] for i in order]
# Example use
docs = [
    {"id":"S1", "text":"xUnit is a unit testing tool for .NET"},
    {"id":"S2", "text":"Cosine similarity compares the angle between vectors"},
    {"id":"S3", "text":"HNSW is a graph based ANN index"},
]
r = Retriever(docs)
print([docs[i]["id"] for i in r.search("How to compare vectors by angle?", k=3, top=2)])